# 03 - Limpieza Silver

En este notebook se integran los datos Bronze, se estandarizan nombres de columnas, se convierten tipos de datos, se eliminan duplicados, se aplican reglas de calidad y se crean columnas calculadas.

In [0]:
from pyspark.sql.functions import (
    col, trim, upper, initcap, current_timestamp,
    when, round, concat_ws, sha2
)

In [0]:
# Configuración general del proyecto

catalog_name = "anemia_ayacucho"

schema_bronze = "bronze"
schema_silver = "silver"

spark.sql(f"USE CATALOG {catalog_name}")

DataFrame[]

In [0]:
# Lectura de tablas Bronze

df_2023 = spark.table(f"{catalog_name}.{schema_bronze}.anemia_2023")
df_2024 = spark.table(f"{catalog_name}.{schema_bronze}.anemia_2024")
df_2025 = spark.table(f"{catalog_name}.{schema_bronze}.anemia_2025")
df_zonas_bronze = spark.table(f"{catalog_name}.{schema_bronze}.maestro_zonas")

print("Registros Bronze 2023:", df_2023.count())
print("Registros Bronze 2024:", df_2024.count())
print("Registros Bronze 2025:", df_2025.count())

Registros Bronze 2023: 549
Registros Bronze 2024: 549
Registros Bronze 2025: 549


In [0]:
# Estandarización inicial de columnas desde Bronze

def estandarizar_anemia(df):
    df_estandar = (
        df.selectExpr(
            "`UBIGEO` as ubigeo",
            "`PROVINCIA` as provincia",
            "`DISTRITO` as distrito",
            "`N° EVALUADOS` as nro_evaluados",
            "`N° NIÑOS CON ANEMIA` as nro_ninos_anemia",
            "`AÑO` as anio",
            "`ZONAS` as zona",
            "fecha_carga",
            "archivo_fuente"
        )
    )
    
    return df_estandar

df_2023_std = estandarizar_anemia(df_2023)
df_2024_std = estandarizar_anemia(df_2024)
df_2025_std = estandarizar_anemia(df_2025)

In [0]:
# Unión de los tres años en una sola tabla operacional

df_anemia_unida = (
    df_2023_std
    .unionByName(df_2024_std)
    .unionByName(df_2025_std)
)

print("Registros unidos antes de limpieza:", df_anemia_unida.count())

display(df_anemia_unida.limit(10))

Registros unidos antes de limpieza: 1647


ubigeo,provincia,distrito,nro_evaluados,nro_ninos_anemia,anio,zona,fecha_carga,archivo_fuente
1,Cangallo,Cangallo,21,6,2023,Zona Norte,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,21,6,2023,Zona Sur,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,21,6,2023,Zona Este,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,21,6,2023,Zona Oeste,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,21,6,2023,Zona Centro,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,21,6,2023,Zona Noreste,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,20,5,2023,Zona Noroeste,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,20,5,2023,Zona Sureste,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,20,5,2023,Zona Suroeste,2026-06-26T06:28:20.042Z,data_2023.csv
2,Cangallo,Chuschi,26,11,2023,Zona Norte,2026-06-26T06:28:20.042Z,data_2023.csv


In [0]:
# Limpieza, conversión de tipos y creación de columnas calculadas

df_anemia_limpia = (
    df_anemia_unida
    .withColumn("ubigeo", trim(col("ubigeo")).cast("int"))
    .withColumn("provincia", initcap(trim(col("provincia"))))
    .withColumn("distrito", initcap(trim(col("distrito"))))
    .withColumn("nro_evaluados", trim(col("nro_evaluados")).cast("int"))
    .withColumn("nro_ninos_anemia", trim(col("nro_ninos_anemia")).cast("int"))
    .withColumn("anio", trim(col("anio")).cast("int"))
    .withColumn("zona", initcap(trim(col("zona"))))
    .withColumn(
        "tasa_anemia",
        round((col("nro_ninos_anemia") / col("nro_evaluados")) * 100, 2)
    )
    .withColumn(
        "nivel_riesgo",
        when(col("tasa_anemia") >= 40, "Alto")
        .when(col("tasa_anemia") >= 20, "Medio")
        .otherwise("Bajo")
    )
    .withColumn("fecha_proceso_silver", current_timestamp())
)

display(df_anemia_limpia.limit(10))

ubigeo,provincia,distrito,nro_evaluados,nro_ninos_anemia,anio,zona,fecha_carga,archivo_fuente,tasa_anemia,nivel_riesgo,fecha_proceso_silver
1,Cangallo,Cangallo,21,6,2023,Zona Norte,2026-06-26T06:28:20.042Z,data_2023.csv,28.57,Medio,2026-06-26T06:30:30.627Z
1,Cangallo,Cangallo,21,6,2023,Zona Sur,2026-06-26T06:28:20.042Z,data_2023.csv,28.57,Medio,2026-06-26T06:30:30.627Z
1,Cangallo,Cangallo,21,6,2023,Zona Este,2026-06-26T06:28:20.042Z,data_2023.csv,28.57,Medio,2026-06-26T06:30:30.627Z
1,Cangallo,Cangallo,21,6,2023,Zona Oeste,2026-06-26T06:28:20.042Z,data_2023.csv,28.57,Medio,2026-06-26T06:30:30.627Z
1,Cangallo,Cangallo,21,6,2023,Zona Centro,2026-06-26T06:28:20.042Z,data_2023.csv,28.57,Medio,2026-06-26T06:30:30.627Z
1,Cangallo,Cangallo,21,6,2023,Zona Noreste,2026-06-26T06:28:20.042Z,data_2023.csv,28.57,Medio,2026-06-26T06:30:30.627Z
1,Cangallo,Cangallo,20,5,2023,Zona Noroeste,2026-06-26T06:28:20.042Z,data_2023.csv,25.0,Medio,2026-06-26T06:30:30.627Z
1,Cangallo,Cangallo,20,5,2023,Zona Sureste,2026-06-26T06:28:20.042Z,data_2023.csv,25.0,Medio,2026-06-26T06:30:30.627Z
1,Cangallo,Cangallo,20,5,2023,Zona Suroeste,2026-06-26T06:28:20.042Z,data_2023.csv,25.0,Medio,2026-06-26T06:30:30.627Z
2,Cangallo,Chuschi,26,11,2023,Zona Norte,2026-06-26T06:28:20.042Z,data_2023.csv,42.31,Alto,2026-06-26T06:30:30.627Z


In [0]:
# Reglas de calidad aplicadas en Silver

registros_antes = df_anemia_limpia.count()

df_anemia_limpia = (
    df_anemia_limpia
    .filter(col("ubigeo").isNotNull())
    .filter(col("provincia").isNotNull())
    .filter(col("distrito").isNotNull())
    .filter(col("zona").isNotNull())
    .filter(col("anio").isin(2023, 2024, 2025))
    .filter(col("nro_evaluados").isNotNull())
    .filter(col("nro_evaluados") > 0)
    .filter(col("nro_ninos_anemia").isNotNull())
    .filter(col("nro_ninos_anemia") >= 0)
    .filter(col("nro_ninos_anemia") <= col("nro_evaluados"))
    .dropDuplicates(["ubigeo", "provincia", "distrito", "anio", "zona"])
)

registros_despues = df_anemia_limpia.count()

print("Registros antes de reglas de calidad:", registros_antes)
print("Registros después de reglas de calidad:", registros_despues)
print("Registros eliminados:", registros_antes - registros_despues)

Registros antes de reglas de calidad: 1647
Registros después de reglas de calidad: 1647
Registros eliminados: 0


In [0]:
# Creación de identificador técnico del registro

df_anemia_limpia = (
    df_anemia_limpia
    .withColumn(
        "id_registro",
        sha2(
            concat_ws(
                "|",
                col("ubigeo").cast("string"),
                col("provincia"),
                col("distrito"),
                col("anio").cast("string"),
                col("zona")
            ),
            256
        )
    )
)

display(df_anemia_limpia.limit(10))

ubigeo,provincia,distrito,nro_evaluados,nro_ninos_anemia,anio,zona,fecha_carga,archivo_fuente,tasa_anemia,nivel_riesgo,fecha_proceso_silver,id_registro
1,Cangallo,Cangallo,21,6,2023,Zona Norte,2026-06-26T06:28:20.042Z,data_2023.csv,28.57,Medio,2026-06-26T06:30:45.224Z,d1b70f172266f7942887034b5ebe55634413b1cfd942d65e3c08a29fa5a1ed73
1,Cangallo,Cangallo,21,6,2023,Zona Sur,2026-06-26T06:28:20.042Z,data_2023.csv,28.57,Medio,2026-06-26T06:30:45.224Z,5a2cfa7061aa91c7c1180046983067757f571b15e6f2e4cd663f8ce36d99df7a
1,Cangallo,Cangallo,21,6,2023,Zona Este,2026-06-26T06:28:20.042Z,data_2023.csv,28.57,Medio,2026-06-26T06:30:45.224Z,1afe8b51dce261b2abfe113eaad53e9836231e046917b3d3db6e009db0944e98
1,Cangallo,Cangallo,21,6,2023,Zona Oeste,2026-06-26T06:28:20.042Z,data_2023.csv,28.57,Medio,2026-06-26T06:30:45.224Z,373dd08fce001aba57ad1e9b833a049a8888cb7ad39c09d592e6afc94979c7bc
1,Cangallo,Cangallo,21,6,2023,Zona Centro,2026-06-26T06:28:20.042Z,data_2023.csv,28.57,Medio,2026-06-26T06:30:45.224Z,8094843a71fab053cfafce084c8174254a8a999b30e5cddd5e4955c1bb9cf233
1,Cangallo,Cangallo,21,6,2023,Zona Noreste,2026-06-26T06:28:20.042Z,data_2023.csv,28.57,Medio,2026-06-26T06:30:45.224Z,b7a23b1cfff73fedc874b7a24ce07d70ab02e0208c9cbd63d1dea01ce50c07c6
1,Cangallo,Cangallo,20,5,2023,Zona Noroeste,2026-06-26T06:28:20.042Z,data_2023.csv,25.0,Medio,2026-06-26T06:30:45.224Z,bfef676bca0c2417e7cb1b9340ad0dc67c6c4138ece3ad58e36b0bfab9750c07
1,Cangallo,Cangallo,20,5,2023,Zona Sureste,2026-06-26T06:28:20.042Z,data_2023.csv,25.0,Medio,2026-06-26T06:30:45.224Z,c9167857bbf2f21f094b02ee9881c8193382c6e80562a44f9336867ee06ffecc
1,Cangallo,Cangallo,20,5,2023,Zona Suroeste,2026-06-26T06:28:20.042Z,data_2023.csv,25.0,Medio,2026-06-26T06:30:45.224Z,f7c04adf1a32a0c8aa5d9d0f25738515cbd96e60ba9df320852cee91b158ad72
2,Cangallo,Chuschi,26,11,2023,Zona Norte,2026-06-26T06:28:20.042Z,data_2023.csv,42.31,Alto,2026-06-26T06:30:45.224Z,ab1b845384746e12fa97e37bb8707fb05e81e48c63b55575d665f458b8cd14f9


In [0]:
# Limpieza del maestro de zonas

df_dim_zona_limpia = (
    df_zonas_bronze
    .select(
        col("id_zona").cast("int").alias("id_zona"),
        initcap(trim(col("zona"))).alias("zona"),
        upper(trim(col("abreviatura"))).alias("abreviatura"),
        initcap(trim(col("tipo_zona"))).alias("tipo_zona"),
        initcap(trim(col("orientacion"))).alias("orientacion"),
        trim(col("descripcion")).alias("descripcion"),
        col("orden_visualizacion").cast("int").alias("orden_visualizacion"),
        col("usar_en_dashboard").cast("boolean").alias("usar_en_dashboard"),
        col("fecha_carga"),
        col("archivo_fuente")
    )
    .dropDuplicates(["zona"])
)

display(df_dim_zona_limpia)

id_zona,zona,abreviatura,tipo_zona,orientacion,descripcion,orden_visualizacion,usar_en_dashboard,fecha_carga,archivo_fuente
1,Zona Norte,N,Cardinal,Norte,Clasificación territorial correspondiente al sector norte del ámbito analizado.,1,true,2026-06-26T06:28:52.730Z,maestro_zonas.json
2,Zona Sur,S,Cardinal,Sur,Clasificación territorial correspondiente al sector sur del ámbito analizado.,2,true,2026-06-26T06:28:52.730Z,maestro_zonas.json
3,Zona Este,E,Cardinal,Este,Clasificación territorial correspondiente al sector este del ámbito analizado.,3,true,2026-06-26T06:28:52.730Z,maestro_zonas.json
4,Zona Oeste,O,Cardinal,Oeste,Clasificación territorial correspondiente al sector oeste del ámbito analizado.,4,true,2026-06-26T06:28:52.730Z,maestro_zonas.json
5,Zona Centro,C,Central,Centro,Clasificación territorial correspondiente al sector central del ámbito analizado.,5,true,2026-06-26T06:28:52.730Z,maestro_zonas.json
6,Zona Noreste,NE,Intercardinal,Noreste,Clasificación territorial correspondiente al sector noreste del ámbito analizado.,6,true,2026-06-26T06:28:52.730Z,maestro_zonas.json
7,Zona Noroeste,NO,Intercardinal,Noroeste,Clasificación territorial correspondiente al sector noroeste del ámbito analizado.,7,true,2026-06-26T06:28:52.730Z,maestro_zonas.json
8,Zona Sureste,SE,Intercardinal,Sureste,Clasificación territorial correspondiente al sector sureste del ámbito analizado.,8,true,2026-06-26T06:28:52.730Z,maestro_zonas.json
9,Zona Suroeste,SO,Intercardinal,Suroeste,Clasificación territorial correspondiente al sector suroeste del ámbito analizado.,9,true,2026-06-26T06:28:52.730Z,maestro_zonas.json


In [0]:
# Validación de zonas entre anemia y maestro de zonas

df_zonas_anemia = df_anemia_limpia.select("zona").distinct()

df_zonas_no_encontradas = (
    df_zonas_anemia
    .join(df_dim_zona_limpia.select("zona"), on="zona", how="left_anti")
)

display(df_zonas_no_encontradas)

zona


In [0]:
# Escritura de tablas Silver en formato Delta

tabla_anemia_silver = f"{catalog_name}.{schema_silver}.anemia_limpia"
tabla_zona_silver = f"{catalog_name}.{schema_silver}.dim_zona_limpia"

(
    df_anemia_limpia.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable(tabla_anemia_silver)
)

(
    df_dim_zona_limpia.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable(tabla_zona_silver)
)

print("Tabla Silver creada:", tabla_anemia_silver)
print("Tabla Silver creada:", tabla_zona_silver)

Tabla Silver creada: anemia_ayacucho.silver.anemia_limpia
Tabla Silver creada: anemia_ayacucho.silver.dim_zona_limpia


In [0]:
# Tabla de reglas de limpieza y calidad aplicadas

reglas_calidad = [
    ("Tratamiento de nulos", "ubigeo, provincia, distrito, zona", "Se eliminaron registros con campos principales nulos."),
    ("Conversión de tipos", "ubigeo, nro_evaluados, nro_ninos_anemia, anio", "Se convirtieron columnas numéricas a tipo entero."),
    ("Estandarización de nombres", "todas las columnas", "Se renombraron columnas originales a formato snake_case."),
    ("Eliminación de duplicados", "ubigeo, provincia, distrito, anio, zona", "Se eliminaron combinaciones repetidas."),
    ("Validación de negocio", "nro_evaluados, nro_ninos_anemia", "Se validó que evaluados sea mayor a cero y que niños con anemia no supere evaluados."),
    ("Columna calculada", "tasa_anemia, nivel_riesgo", "Se calculó la tasa de anemia y una clasificación de riesgo.")
]

df_reglas_calidad = spark.createDataFrame(
    reglas_calidad,
    ["regla", "campo_afectado", "accion_realizada"]
)

(
    df_reglas_calidad.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{schema_silver}.reglas_calidad")
)

display(df_reglas_calidad)

regla,campo_afectado,accion_realizada
Tratamiento de nulos,"ubigeo, provincia, distrito, zona",Se eliminaron registros con campos principales nulos.
Conversión de tipos,"ubigeo, nro_evaluados, nro_ninos_anemia, anio",Se convirtieron columnas numéricas a tipo entero.
Estandarización de nombres,todas las columnas,Se renombraron columnas originales a formato snake_case.
Eliminación de duplicados,"ubigeo, provincia, distrito, anio, zona",Se eliminaron combinaciones repetidas.
Validación de negocio,"nro_evaluados, nro_ninos_anemia",Se validó que evaluados sea mayor a cero y que niños con anemia no supere evaluados.
Columna calculada,"tasa_anemia, nivel_riesgo",Se calculó la tasa de anemia y una clasificación de riesgo.


In [0]:
# Validación de tablas creadas en Silver

display(spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_silver}"))

database,tableName,isTemporary
silver,anemia_limpia,false
silver,dim_zona_limpia,false
silver,reglas_calidad,false


## Evidencia esperada

En este notebook se debe capturar:

- Unión de archivos 2023, 2024 y 2025.
- Renombramiento de columnas.
- Conversión de tipos de datos.
- Cálculo de `tasa_anemia`.
- Reglas de calidad aplicadas.
- Tablas creadas en Silver.